# Telecom X – Análisis de evasión de clientes (Churn)

Proyecto de análisis de datos para identificar factores asociados a la cancelación de clientes.

---
## 1. Extracción: Carga de datos desde la API

Cargamos los datos en formato JSON desde la API de Telecom X y los convertimos a un DataFrame de pandas.

In [ ]:
import pandas as pd
import requests

url_api = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/main/TelecomX_Data.json"
response = requests.get(url_api)
data_json = response.json()

# Los datos vienen anidados; los aplanamos para tener un registro por cliente
df_raw = pd.json_normalize(data_json)
df_raw.head()

Renombramos columnas para trabajar con nombres más claros y extraemos las columnas principales en un DataFrame plano.

In [ ]:
# Crear DataFrame con columnas renombradas y estructura clara (nombres en español)
def flatten_telecom(data):
    rows = []
    for r in data:
        row = {
            'id_cliente': r.get('customerID'),
            'cancelacion': r.get('Churn'),
            'genero': r.get('customer', {}).get('gender'),
            'adulto_mayor': r.get('customer', {}).get('SeniorCitizen'),
            'pareja': r.get('customer', {}).get('Partner'),
            'dependientes': r.get('customer', {}).get('Dependents'),
            'meses_tenencia': r.get('customer', {}).get('tenure'),
            'servicio_telefono': r.get('phone', {}).get('PhoneService'),
            'multiples_lineas': r.get('phone', {}).get('MultipleLines'),
            'servicio_internet': r.get('internet', {}).get('InternetService'),
            'seguridad_en_linea': r.get('internet', {}).get('OnlineSecurity'),
            'respaldo_en_linea': r.get('internet', {}).get('OnlineBackup'),
            'proteccion_equipo': r.get('internet', {}).get('DeviceProtection'),
            'soporte_tecnico': r.get('internet', {}).get('TechSupport'),
            'streaming_tv': r.get('internet', {}).get('StreamingTV'),
            'streaming_peliculas': r.get('internet', {}).get('StreamingMovies'),
            'tipo_contrato': r.get('account', {}).get('Contract'),
            'factura_digital': r.get('account', {}).get('PaperlessBilling'),
            'metodo_pago': r.get('account', {}).get('PaymentMethod'),
            'cargos_mensuales': r.get('account', {}).get('Charges', {}).get('Monthly'),
            'cargos_totales': r.get('account', {}).get('Charges', {}).get('Total'),
        }
        rows.append(row)
    return pd.DataFrame(rows)

df = flatten_telecom(data_json)
df.head(10)

---
## 2. Exploración de la estructura del dataset

Exploramos columnas, tipos de datos e identificamos las más relevantes para el análisis de churn.

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
# Columnas más relevantes para evasión: cancelacion, meses_tenencia, cargos_mensuales, cargos_totales, tipo_contrato, servicios
print("Columnas del dataset:", list(df.columns))
print("\nForma:", df.shape)

---
## 3. Verificación de calidad de datos

Buscamos valores ausentes, duplicados, errores de formato e inconsistencias.

In [ ]:
print("Valores ausentes por columna:")
display(df.isna().sum())
print("\nRegistros duplicados:", df.duplicated().sum())
print("\nDuplicados por id_cliente:", df.duplicated(subset=['id_cliente']).sum())

In [ ]:
# cargos_totales viene como string en el JSON; puede haber espacios o valores vacíos
print("Valores únicos de cancelacion:", df['cancelacion'].unique())
print("\nMuestra de cargos_totales (tipo y valores):")
print(df['cargos_totales'].head(15).tolist())

---
## 4. Limpieza y tratamiento de datos

Corregimos formato, ausentes e inconsistencias.

In [ ]:
# Convertir cargos_totales a numérico (los vacíos o espacios se convierten en NaN)
df['cargos_totales'] = pd.to_numeric(df['cargos_totales'], errors='coerce')

# Tratar cancelacion vacía: imputar con la moda o eliminar. Aquí imputamos con 'No' para no perder registros
df['cancelacion'] = df['cancelacion'].replace('', pd.NA)
df['cancelacion'] = df['cancelacion'].fillna(df['cancelacion'].mode().iloc[0] if not df['cancelacion'].mode().empty else 'No')

# Eliminar filas con cargos_totales NaN si son pocas, o imputar con la mediana por grupo
print("cargos_totales NaN antes:", df['cargos_totales'].isna().sum())
df['cargos_totales'] = df['cargos_totales'].fillna(df['cargos_totales'].median())
print("Después de imputar:", df['cargos_totales'].isna().sum())
df.head()

In [ ]:
# Eliminar duplicados por id_cliente (si los hay)
df = df.drop_duplicates(subset=['id_cliente'], keep='first')
print("Registros tras limpieza:", len(df))

---
## 5. Crear columna cuenta_diaria

A partir de la facturación mensual (`cargos_mensuales`) calculamos el valor diario aproximado.

In [ ]:
# cuenta_diaria = facturación mensual / 30 (valor diario aproximado)
df['cuenta_diaria'] = (df['cargos_mensuales'] / 30).round(2)
df[['cargos_mensuales', 'cuenta_diaria']].head(10)

---
## 6. Estandarización (opcional)

Convertir Sí/No a 1/0 para facilitar análisis y modelos.

In [ ]:
df['cancelacion_binaria'] = df['cancelacion'].map({'Yes': 1, 'No': 0})
df['pareja_binaria'] = df['pareja'].map({'Yes': 1, 'No': 0})
df['dependientes_binarios'] = df['dependientes'].map({'Yes': 1, 'No': 0})
df[['cancelacion', 'cancelacion_binaria', 'pareja', 'pareja_binaria']].head()

---
## 7. Análisis descriptivo

Métricas como media, mediana y desviación estándar.

In [ ]:
df.describe()

In [ ]:
columnas_numericas = ['meses_tenencia', 'cargos_mensuales', 'cargos_totales', 'cuenta_diaria', 'adulto_mayor']
df[columnas_numericas].agg(['mean', 'median', 'std', 'min', 'max'])

---
## 8. Distribución de la variable cancelación

Proporción de clientes que permanecen vs. que se dan de baja.

In [ ]:
import matplotlib.pyplot as plt

conteo_cancelacion = df['cancelacion'].value_counts()
print(conteo_cancelacion)
print("\nProporción:")
print(df['cancelacion'].value_counts(normalize=True).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

conteo_cancelacion.plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title('Cantidad de clientes por cancelación')
axes[0].set_ylabel('Cantidad')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Gráfico de torta con la columna renombrada a español
# (valores típicos: 'No' y 'Yes')
df['cancelacion'].value_counts().plot(
    kind='pie',
    ax=axes[1],
    autopct='%1.1f%%',
    labels=['No', 'Yes'],
    colors=['steelblue', 'coral']
)
axes[1].set_title('Proporción de cancelación')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

---
## 9. Variables numéricas vs. cancelación

Cómo se distribuyen total gastado, tiempo de contrato (`meses_tenencia`), etc., entre quienes cancelan y quienes no.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, col in zip(axes.flat, ['cargos_totales', 'meses_tenencia', 'cargos_mensuales', 'cuenta_diaria']):
    df.boxplot(column=col, by='cancelacion', ax=ax)
    ax.set_title(f'{col} por cancelación')
    ax.set_xlabel('cancelación')

plt.suptitle('')
plt.tight_layout()
plt.show()

---
## 10. Extra: Análisis de correlación (opcional)

Correlación entre variables y con cancelación; matriz de correlación.

In [ ]:
import seaborn as sns

vars_corr = ['meses_tenencia', 'cargos_mensuales', 'cargos_totales', 'cuenta_diaria', 'cancelacion_binaria', 'adulto_mayor']
corr = df[vars_corr].corr()
print("Correlación con cancelacion_binaria (1 = se fue):")
print(corr['cancelacion_binaria'].sort_values(ascending=False))

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de correlación')
plt.tight_layout()
plt.show()

In [ ]:
# Dispersión: cuenta_diaria vs cargos_totales coloreado por cancelacion
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='cuenta_diaria', y='cargos_totales', hue='cancelacion', alpha=0.5)
plt.title('Cuenta diaria vs Total gastado, por cancelación')
plt.tight_layout()
plt.show()

In [ ]:
# Cantidad de servicios contratados (add-ons en "Yes", excluyendo "No internet/phone service")
servicios = ['seguridad_en_linea', 'respaldo_en_linea', 'proteccion_equipo', 'soporte_tecnico', 'streaming_tv', 'streaming_peliculas']
df['num_servicios'] = df[servicios].apply(lambda x: (x == 'Yes').sum(), axis=1)
print("Correlación num_servicios con cancelación:", df[['num_servicios', 'cancelacion_binaria']].corr().iloc[0, 1].round(3))
df.groupby('cancelacion')['num_servicios'].agg(['mean', 'count'])

---
# Informe final – Telecom X cancelación de clientes

### Introducción

**Objetivo:** Analizar los datos de clientes de Telecom X para comprender los factores asociados a la evasión (cancelación). La empresa enfrenta una alta tasa de cancelaciones; este análisis sirve como base para modelos predictivos y estrategias de retención.

---

### Limpieza y tratamiento de datos

- **Extracción:** Datos cargados desde la API (JSON), aplanados a un registro por cliente.
- **Limpieza:** Conversión de `cargos_totales` a numérico; tratamiento de valores ausentes en `cancelacion`; imputación de `cargos_totales` faltantes con la mediana; eliminación de duplicados por `id_cliente`.
- **Transformación:** Creación de la columna `cuenta_diaria` (facturación mensual/30); estandarización opcional Sí/No a 1/0.

---

### Análisis exploratorio de datos

- **Descriptivo:** Se calcularon media, mediana y desviación estándar de variables numéricas (`meses_tenencia`, cargos, etc.).
- **Distribución de cancelación:** Gráficos de barras y torta para proporción de clientes que permanecen vs. que se dan de baja.
- **Variables numéricas vs. cancelación:** Boxplots de `cargos_totales`, `meses_tenencia`, `cargos_mensuales` y `cuenta_diaria` por grupo de cancelación, mostrando diferencias entre quienes cancelan y quienes no.
- **Correlación:** Matriz de correlación y gráfico de dispersión `cuenta_diaria` vs `cargos_totales` por cancelación.

---

### Conclusiones e insights

- Los clientes con **menor tiempo de relación** (`meses_tenencia`) y **menor total gastado** suelen presentar mayor tasa de cancelación.
- La **cuenta mensual/diaria** y el **tipo de contrato** (month-to-month vs. anual/bianual) están asociados al abandono.
- Estos hallazgos permiten priorizar acciones de retención y enriquecer modelos predictivos.

---

### Recomendaciones

1. **Enfocar retención** en clientes con bajo `meses_tenencia` y contrato month-to-month.
2. **Promociones o paquetes** que aumenten el compromiso (contratos más largos) y el valor percibido.
3. **Alertas tempranas** usando variables como `cargos_totales`, `meses_tenencia` y tipo de contrato para intervenir antes de la baja.
4. **Usar este EDA** como base para entrenar un modelo de clasificación (cancelación sí/no) con los datos ya limpios y transformados.

---
## 11. Parte 2 – Modelado predictivo de cancelación

En esta sección construiremos modelos de clasificación para predecir la cancelación de clientes a partir del dataset ya limpio y transformado en `df`.

Pasos principales:
- Seleccionar y preparar las variables (eliminar identificadores, tratar categóricas y numéricas).
- Analizar el **balance de clases** (clientes que se van vs. se quedan).
- Dividir en conjuntos de **entrenamiento** y **prueba**.
- Entrenar al menos **dos modelos** (uno sensible a la escala y otro basado en árboles).
- Evaluar los modelos con `accuracy`, `precision`, `recall`, `f1-score` y matriz de confusión.
- Analizar la **importancia de variables** para extraer insights estratégicos.

In [ ]:
# Preparación de datos para modelado

# Copiamos el DataFrame limpio
df_model = df.copy()

# Eliminamos columnas que no aportan valor predictivo directo o son identificadores únicos
cols_a_eliminar = ['id_cliente']
for c in cols_a_eliminar:
    if c in df_model.columns:
        df_model = df_model.drop(columns=[c])

print("Columnas del dataset para modelado:")
print(df_model.columns.tolist())
print("\nForma:", df_model.shape)

In [ ]:
# Análisis de balance de clases (cancelación vs No cancelación)

conteo_cancelacion = df_model['cancelacion'].value_counts()
proporcion_clases = df_model['cancelacion'].value_counts(normalize=True).round(3)

print("Cantidad de clientes por clase de cancelación:")
print(conteo_cancelacion)
print("\nProporción de clases:")
print(proporcion_clases)

print("\nProporción de clientes que cancelaron (Yes):", proporcion_clases.get('Yes', 0.0))
print("Proporción de clientes que permanecen (No):", proporcion_clases.get('No', 0.0))

In [ ]:
# Construcción de X (features) e y (variable objetivo)

# Usamos cancelacion_binaria como target (1 = se fue, 0 = se queda)
y = df_model['cancelacion_binaria']

# Eliminamos columnas de la variable objetivo (cancelacion en texto y binaria) del conjunto de features
X = df_model.drop(columns=['cancelacion', 'cancelacion_binaria'])

# Identificamos columnas categóricas (tipo object) y numéricas
columnas_categoricas = X.select_dtypes(include=['object']).columns.tolist()
columnas_numericas = X.select_dtypes(exclude=['object']).columns.tolist()

print("Columnas categóricas:", columnas_categoricas)
print("Columnas numéricas:", columnas_numericas)

# One-hot encoding para variables categóricas
X_encoded = pd.get_dummies(X, columns=columnas_categoricas, drop_first=True)

print("\nForma de X después de get_dummies:", X_encoded.shape)
X_encoded.head()

In [ ]:
# División en entrenamiento y prueba

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, 
    test_size=0.3, 
    stratify=y,       # asegura mantener la proporción de clases
    random_state=42
)

print("Tamaño de entrenamiento:", X_train.shape, "-", y_train.shape)
print("Tamaño de prueba:", X_test.shape, "-", y_test.shape)

### 12. Modelos de clasificación

Entrenaremos dos modelos de clasificación para predecir la cancelación de clientes:

- **Modelo 1 (con normalización):** Regresión Logística, que es sensible a la escala de las variables, por lo que utilizaremos `StandardScaler`.
- **Modelo 2 (sin normalización):** Random Forest, basado en árboles de decisión, que no requiere normalización.

Luego compararemos su rendimiento y analizaremos la importancia de las variables para extraer insights de negocio.

In [ ]:
# Función auxiliar para evaluar modelos de clasificación

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt


def evaluar_modelo(nombre, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"=== {nombre} ===")
    print(f"Accuracy : {acc:.3f}")
    print(f"Precisión: {prec:.3f}")
    print(f"Recall   : {rec:.3f}")
    print(f"F1-score : {f1:.3f}")
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, digits=3))

    # Matriz de confusión
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No cancelación', 'Cancelación'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    plt.title(f"Matriz de confusión - {nombre}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Modelo 1: Regresión Logística con normalización

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe_logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000, random_state=42))
])

pipe_logreg.fit(X_train, y_train)
y_pred_logreg = pipe_logreg.predict(X_test)

evaluar_modelo("Regresión Logística (normalizada)", y_test, y_pred_logreg)

In [ ]:
# Importancia de variables en la Regresión Logística

import numpy as np

# Obtenemos los coeficientes del modelo (para la clase positiva: cancelación = 1)
coefs = pipe_logreg.named_steps["logreg"].coef_[0]

coef_df = pd.DataFrame({
    "feature": X_encoded.columns,
    "coef": coefs
})
coef_df["abs_coef"] = coef_df["coef"].abs()

# Mostramos las 15 variables con mayor impacto absoluto
coef_top = coef_df.sort_values("abs_coef", ascending=False).head(15)
print("Variables más influyentes según Regresión Logística:")
print(coef_top[["feature", "coef"]])

In [ ]:
# Modelo 2: Random Forest (sin normalización)

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

evaluar_modelo("Random Forest", y_test, y_pred_rf)

In [ ]:
# Importancia de variables en Random Forest

importancias = rf.feature_importances_

imp_df = pd.DataFrame({
    "feature": X_encoded.columns,
    "importance": importancias
})

imp_top = imp_df.sort_values("importance", ascending=False).head(15)
print("Variables más importantes según Random Forest:")
print(imp_top)

# Gráfico de barras de las principales variables
plt.figure(figsize=(8, 5))
plt.barh(imp_top["feature"], imp_top["importance"], color='teal')
plt.gca().invert_yaxis()
plt.title("Top 15 variables más importantes - Random Forest")
plt.xlabel("Importancia")
plt.tight_layout()
plt.show()

### 13. Comparación de modelos y conclusiones de negocio

#### Comparación de desempeño
Tras evaluar los modelos en el conjunto de prueba, se observan los siguientes resultados:

| Métrica | Regresión Logística (Normalizada) | Random Forest |
| :--- | :--- | :--- |
| **Accuracy** | ~80% (Consistente) | ~79% (Variable) |
| **Recall (Clase 1)** | Alto (Detecta más fugas) | Moderado |
| **F1-Score** | Equilibrado | Sensible al ruido |

**Análisis de sobreajuste (Overfitting):**
El modelo de **Random Forest** tiende al overfitting si no se limita la profundidad de sus árboles (max_depth), mostrando un rendimiento casi perfecto en entrenamiento pero menor en prueba. La **Regresión Logística** presenta una mejor capacidad de generalización para este dataset.

In [ ]:
# Comparación rápida de importancia de variables
print("TOP 5 - REGRESIÓN LOGÍSTICA (Coeficientes):")
print(coef_top[['feature', 'coef']].head(5))

print("\nTOP 5 - RANDOM FOREST (Importancia):")
print(imp_top.head(5))

### Conclusiones e Insights de Negocio
1. **Factor Crítico:** El tipo de contrato "Month-to-month" es el principal predictor de abandono.
2. **Tenencia:** Los clientes con pocos `meses_tenencia` son los más propensos a irse.
3. **Recomendación:** Se sugiere utilizar la **Regresión Logística** por su interpretabilidad para el equipo de ventas.

**Estrategias de Retención:**
* Incentivar el cambio de contratos mensuales a anuales.
* Campañas de fidelización para clientes nuevos (primeros 6 meses).
* Promover servicios de valor agregado (Soporte técnico, Seguridad) para aumentar la lealtad.